In [1]:
import requests
from bs4 import BeautifulSoup  
import time                
import pandas as pd

In [2]:
# Base URL for the practice site.
BASE_URL = "https://www.nhsinform.scot/illnesses-and-conditions/a-to-z/"

# 'HEADERS' is a dictionary telling the website who we are.
# Many sites reject requests without a 'User-Agent'.
HEADERS = {                                           
    "User-Agent": (                                   
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "  
        "AppleWebKit/537.36 (KHTML, like Gecko) "     
        "Chrome/119.0.0.0 Safari/537.36"              
    )
}

In [4]:
def fetch_html(url, part="all", remove="none"):
    """
    Fetch HTML from a URL and return a BeautifulSoup object.

    Parameters:
        url (str): The target URL to fetch.
        part (str): Which part of the HTML to return — 'all', 'head', or 'body'. Default = 'all'.
        remove (str): What to remove — 'none', 'script', 'style', or 'both'. Default = 'none'.

    Returns:
        BeautifulSoup object or None (on failure)
    """
    try:
        resp = requests.get(url, headers=HEADERS, timeout=15)  # Send GET request
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")     # Parse HTML

            # Select part of the document
            if part == "head":
                section = soup.head
            elif part == "body":
                section = soup.body
            else:
                section = soup

            # Remove unwanted tags
            if remove in ("script", "both"):
                for s in section.find_all("script"):
                    s.decompose()
            if remove in ("style", "both"):
                for s in section.find_all("style"):
                    s.decompose()

            return section

        else:
            print(f"[warn] {url} -> HTTP {resp.status_code}")
            return None

    except requests.RequestException as e:
        print(f"[error] Request failed for {url}: {e}")
        return None

In [5]:
# ------------------------------------
# PARSE INDIVIDUAL ILLNESS PAGE
# ------------------------------------
def parse_illness_page(soup):
    """Extract all <h2> sections with their following <p> paragraphs."""
    data = {}
    if soup is None:
        return data

    for h2 in soup.find_all("h2"):
        title = h2.get_text(strip=True)
        paragraphs = []
        for sib in h2.find_next_siblings():
            if sib.name == "h2":
                break
            if sib.name == "p":
                paragraphs.append(sib.get_text(strip=True))
        if paragraphs:
            data[title] = " ".join(paragraphs)

    return data


In [ ]:
def parse_page(soup):                                             # Define function to extract quotes from one page
    # Find all <a> tags inside the div
    links = soup.select("div.az_list_indivisual a")
    
    # Extract text and URLs
    for a in links:
        text = a.get_text(strip=True)
        target_url = a["href"]
        
        print("Text:", text)
        print("URL:", target_url) 

        htmlsoup = fetch_html(target_url, "body", remove="script")
        illness_data = parse_illness_page(htmlsoup)
        print(illness_data)

In [6]:
from pymongo import MongoClient
import time

def get_mongo_collection(collection_name="Illnesses"):
    """
    Connect to MongoDB and return a specific collection.
    Default = Illnesses.
    """
    client = MongoClient("mongodb://localhost:27017/")
    db = client["Medical_Diagnosis"]  # Database name
    return db[collection_name]


def parse_page(soup):
    """
    Parse the A–Z listing page:
      - Extract illness names + URLs
      - Scrape each illness page
      - Parse its <h2> sections and <p> descriptions
      - Insert each illness into MongoDB
    """
    collection = get_mongo_collection("Illnesses")
    links = soup.select("div.az_list_indivisual a")
    
    all_docs = []

    for a in links:
        text = a.get_text(strip=True)
        target_url = a["href"]
        if not target_url.startswith("http"):
            target_url = "https://www.nhsinform.scot" + target_url

        print(f"\n🔎 Scraping: {text}")
        print(f"🔗 URL: {target_url}")

        htmlsoup = fetch_html(target_url, "body", remove="script")
        illness_data = parse_illness_page(htmlsoup)

        if not illness_data:
            print("⚠️ Skipped (no data found)")
            continue

        doc = {
            "illness_name": text,
            "url": target_url,
            "sections": [
                {"section_title": k, "description": v}
                for k, v in illness_data.items()
            ],
            "scraped_at": time.strftime("%Y-%m-%d %H:%M:%S")
        }

        all_docs.append(doc)
        print(f"✅ Parsed: {text}")

        # Pause to avoid hammering the server
        time.sleep(1.5)

    # ✅ Insert all documents at once into MongoDB
    if all_docs:
        try:
            collection.insert_many(all_docs, ordered=False)
            print(f"\n✅ Inserted {len(all_docs)} illness records into MongoDB collection 'Illnesses'")
        except Exception as e:
            print(f"⚠️ Mongo insert error: {e}")
    else:
        print("\n⚠️ No valid data found to insert.")


In [7]:
htmlsoup = fetch_html(BASE_URL, "body", remove="script")
parse_page(htmlsoup)


🔎 Scraping: Abdominal aortic aneurysm
🔗 URL: https://www.nhsinform.scot/illnesses-and-conditions/a-to-z/abdominal-aortic-aneurysm/
✅ Parsed: Abdominal aortic aneurysm

🔎 Scraping: Achilles tendinopathy
🔗 URL: https://www.nhsinform.scot/illnesses-and-conditions/muscle-bone-and-joints/leg-and-foot-problems-and-conditions/achilles-tendinopathy/
✅ Parsed: Achilles tendinopathy

🔎 Scraping: Acne
🔗 URL: https://www.nhsinform.scot/illnesses-and-conditions/skin-hair-and-nails/acne/
✅ Parsed: Acne

🔎 Scraping: Acute cholecystitis
🔗 URL: https://www.nhsinform.scot/illnesses-and-conditions/stomach-liver-and-gastrointestinal-tract/acute-cholecystitis/
[warn] https://www.nhsinform.scot/illnesses-and-conditions/stomach-liver-and-gastrointestinal-tract/acute-cholecystitis/ -> HTTP 403
⚠️ Skipped (no data found)

🔎 Scraping: Acute lymphoblastic leukaemia
🔗 URL: https://www.nhsinform.scot/illnesses-and-conditions/cancer/cancer-types-in-adults/acute-lymphoblastic-leukaemia/
✅ Parsed: Acute lymphoblasti

In [ ]:
# ------------------------------------
# PARSE INDIVIDUAL ILLNESS PAGE
# ------------------------------------
#def parse_illness_page(soup):
 #   """Extract all <h2> sections with their following <p> paragraphs."""
  #  data = {}
   # if soup is None:
    #    return data

    #for h2 in soup.find_all("h2"):
     #   title = h2.get_text(strip=True)
      #  paragraphs = []
       # for sib in h2.find_next_siblings():
        #    if sib.name == "h2":
         #       break
          #  if sib.name == "p":
           #     paragraphs.append(sib.get_text(strip=True))
       # if paragraphs:
        #    data[title] = " ".join(paragraphs)

    #return data
#illness_data = parse_illness_page(htmlsoup)
#illness_data
